# 18.1 教師能提供什麼學習訊號？

# 第 18 章：蒸餾，讓小模型向教師學習

前置：CE、模型輸出與學生訓練。教師像經驗較多的老師：可以只給答案，也可展示對各候選的分配。老師可能說錯；學生也須真的選較小架構。此處只驗loss與梯度，正式蒸餾需讀者提供已訓練教師。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：老師不只說第一名，也指出接近的候選**

學生選較小架構；老師可能教錯，仍要真值與行爲評估。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/distillation.svg")))

**先想一想：**只拿到老師最終文字，能知道所有候選的概率嗎？

教師可以給整段文字供SFT，也可以給每個位置的候選分佈。兩種信號的接口、成本與適用tokenizer不同，先分清楚。

**把直覺寫成數學：**黑盒=generated answers；白盒=teacher logits/probabilities。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
signals = {"answer": "4", "distribution": {"4": 0.8, "3": 0.15, "5": 0.05}}
print(signals)
print("答案來自生成；分佈需要模型forward訪問，兩者不能互相冒充")

**如何讀結果：**示例分佈是手設，保持和真實teacher結果區分。

**只改一件事：**只保留answer字段，指出哪些蒸餾loss不能再用。
